In [1]:
import torch
import torch.nn as nn


# ── nn.Embedding — LOOKUP TABLE FOR TOKEN VECTORS ────────────────────────────
# Algorithm    : integer index → dense float vector via matrix row lookup
# embedding    : weight matrix of shape (vocab_size, d_model)
#                each row i is the learned vector for token ID i
# input_ids    : integer tensor of token IDs, any shape (...,)
# output       : same shape as input but with d_model appended at the end
#                input (...,) → output (..., d_model)
# Precondition : every ID in input_ids must be in [0, vocab_size)
# Time  : O(seq_len · d_model) — just row indexing, no matmul
# Space : O(vocab_size · d_model) for the weight matrix
#         GPT-2 : 50257 × 768 = 38,597,376 floats ≈ 147 MB at float32
# ─────────────────────────────────────────────────────────────────────────────

vocab_size = 50257
d_model    = 768

embedding = nn.Embedding(vocab_size, d_model)
# embedding.weight shape : (50257, 768)
# initialized by default with N(0, 1) — learned during training via backprop

print("embedding weight shape :", embedding.weight.shape)
print("total parameters       :", embedding.weight.numel())
print("memory (float32, MB)   :", round(embedding.weight.numel() * 4 / 1024**2, 2))
# embedding weight shape : torch.Size([50257, 768])
# total parameters       : 38597376
# memory (float32, MB)   : 147.19


# ── FORWARD PASS ─────────────────────────────────────────────────────────────
input_ids = torch.tensor([[15496, 11, 314, 716, 4673, 48108, 68, 0]])
# shape : (1, 8)
# these are GPT-2 token IDs for "Hello, I am learning Transformers!"
# each integer is a row index into embedding.weight

embedded = embedding(input_ids)
# shape : (1, 8, 768)
# internally : for each id in input_ids, return embedding.weight[id]
# equivalent to : embedding.weight[input_ids]  ← pure row indexing

print("\nforward pass")
print("input_ids shape :", input_ids.shape)
print("embedded shape  :", embedded.shape)
print("token 0 id      :", input_ids[0, 0].item(), "→ vector[:5]", embedded[0, 0, :5].tolist())
print("token 1 id      :", input_ids[0, 1].item(), "→ vector[:5]", embedded[0, 1, :5].tolist())
# input_ids shape : torch.Size([1, 8])
# embedded shape  : torch.Size([1, 8, 768])
# token 0 id      : 15496 → vector[:5] [x.xx, x.xx, ...]
# token 1 id      : 11    → vector[:5] [x.xx, x.xx, ...]


# ── WHAT nn.Embedding IS DOING UNDER THE HOOD ────────────────────────────────
# nn.Embedding(V, D) is just a (V, D) float matrix with row-index lookup.
# There is NO matmul. It is equivalent to:

W = embedding.weight                          # (50257, 768)
manual_lookup = W[input_ids]                  # (1, 8, 768) — fancy indexing
matches = torch.allclose(manual_lookup, embedded)

print("\nmanual W[input_ids] matches embedding(input_ids) :", matches)
# True — they are identical; embedding() is syntactic sugar for row indexing


# ── GRADIENT FLOW ─────────────────────────────────────────────────────────────
# During backprop only the rows that were looked up get gradients.
# Rows for tokens not in the batch get zero gradient that step.
# This is why sparse_grad=True (optional) can save memory for large vocabs.

print("\nembedding.weight.requires_grad :", embedding.weight.requires_grad)
# True — weight matrix is a learnable parameter


# ── PADDING TOKEN ─────────────────────────────────────────────────────────────
# padding_idx : tells embedding to keep that row as all-zeros always
#               and its gradient is zeroed out — pad tokens contribute nothing

embedding_with_pad = nn.Embedding(vocab_size, d_model, padding_idx=0)
pad_vector = embedding_with_pad(torch.tensor([0]))

print("\npadding_idx=0 vector all zeros :", torch.all(pad_vector == 0).item())
# True — row 0 is permanently zeroed; no gradient flows through it


# ── SMALL WORKED EXAMPLE FROM FIRST PRINCIPLES ───────────────────────────────
# Build a tiny embedding by hand to see exactly what happens

V, D = 5, 3   # vocab of 5 tokens, vectors of dim 3
W_small = torch.tensor([
    [0.1, 0.2, 0.3],   # token 0
    [0.4, 0.5, 0.6],   # token 1
    [0.7, 0.8, 0.9],   # token 2
    [1.0, 1.1, 1.2],   # token 3
    [1.3, 1.4, 1.5],   # token 4
])

emb_small = nn.Embedding(V, D)
emb_small.weight = nn.Parameter(W_small)   # inject known weights

ids   = torch.tensor([[2, 0, 4]])          # batch=1, seq_len=3
out   = emb_small(ids)                     # (1, 3, 3)

print("\nsmall embedding worked example")
print("W_small :\n", W_small)
print("input ids     :", ids.tolist())
print("expected row 2:", W_small[2].tolist())
print("expected row 0:", W_small[0].tolist())
print("expected row 4:", W_small[4].tolist())
print("output[0]     :\n", out[0])
# output[0] must equal stacking rows [2, 0, 4] from W_small

assert torch.allclose(out[0, 0], W_small[2])
assert torch.allclose(out[0, 1], W_small[0])
assert torch.allclose(out[0, 2], W_small[4])
print("all row lookups verified")
# all row lookups verified


# ── DRY RUN : embedding(input_ids) for input_ids = [[2, 0, 4]] ──────────────
# State  : W_small = 5×3 matrix (rows = token vectors)
#          input_ids = [[2, 0, 4]]  shape (1, 3)
#
# Action : for batch 0, position 0 → id=2 → return W_small[2] = [0.7, 0.8, 0.9]
#          for batch 0, position 1 → id=0 → return W_small[0] = [0.1, 0.2, 0.3]
#          for batch 0, position 2 → id=4 → return W_small[4] = [1.3, 1.4, 1.5]
#
# Stack results along seq_len dim → shape (1, 3, 3)
# New state : out[0] = [[0.7, 0.8, 0.9],
#                        [0.1, 0.2, 0.3],
#                        [1.3, 1.4, 1.5]]

embedding weight shape : torch.Size([50257, 768])
total parameters       : 38597376
memory (float32, MB)   : 147.24

forward pass
input_ids shape : torch.Size([1, 8])
embedded shape  : torch.Size([1, 8, 768])
token 0 id      : 15496 → vector[:5] [-0.6856263279914856, 1.3392900228500366, -0.6079111099243164, 1.2227543592453003, -0.45432984828948975]
token 1 id      : 11 → vector[:5] [1.2311091423034668, -1.031427025794983, 0.6441848874092102, 1.1785752773284912, -0.8805479407310486]

manual W[input_ids] matches embedding(input_ids) : True

embedding.weight.requires_grad : True

padding_idx=0 vector all zeros : True

small embedding worked example
W_small :
 tensor([[0.1000, 0.2000, 0.3000],
        [0.4000, 0.5000, 0.6000],
        [0.7000, 0.8000, 0.9000],
        [1.0000, 1.1000, 1.2000],
        [1.3000, 1.4000, 1.5000]])
input ids     : [[2, 0, 4]]
expected row 2: [0.699999988079071, 0.800000011920929, 0.8999999761581421]
expected row 0: [0.10000000149011612, 0.20000000298023224, 0.

In [1]:
import torch
import torch.nn.functional as F
import math


# ── SCALED DOT-PRODUCT ATTENTION — FROM FIRST PRINCIPLES ─────────────────────
# Algorithm    : Attention(Q, K, V) = softmax(QKᵀ / √d_k) · V
# Q            : query matrix  — "what am I looking for?"
# K            : key matrix    — "what do I contain?"
# V            : value matrix  — "what do I return if selected?"
# d_k          : head dimension; √d_k scaling prevents dot products from
#                growing too large in magnitude → keeps softmax in gradient-friendly zone
# scores       : (batch, heads, seq_len_q, seq_len_k) — raw similarity of every Q to every K
# weights      : scores after softmax → probability distribution over key positions
# output       : weighted sum of V rows; each query position gets a blend of all values
# Precondition : Q, K, V must have same d_k; K and V must have same seq_len
# Time  : O(seq_len² · d_k) — the seq_len² is the bottleneck (quadratic in sequence)
# Space : O(seq_len²) for the attention weight matrix
# ─────────────────────────────────────────────────────────────────────────────

def scaled_dot_product_attention(Q, K, V, attention_mask=None):
    d_k = Q.size(-1)                                   # head dimension

    # Step 1 : raw similarity scores
    # K.transpose(-2, -1) : (batch, heads, d_k, seq_len_k)
    # matmul result       : (batch, heads, seq_len_q, seq_len_k)
    # each entry [b,h,i,j] = dot(Q[b,h,i,:], K[b,h,j,:]) / √d_k
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    # Step 2 : mask padding positions
    # attention_mask == 0 marks [PAD] tokens
    # fill with -1e9 so softmax output ≈ 0 for those positions (e^-1e9 ≈ 0)
    if attention_mask is not None:
        scores = scores.masked_fill(attention_mask == 0, -1e9)

    # Step 3 : softmax over key dimension → attention weights
    # dim=-1 : each query position i produces a distribution over all key positions j
    weights = F.softmax(scores, dim=-1)

    # Step 4 : weighted sum of value rows
    # weights : (batch, heads, seq_len_q, seq_len_k)
    # V       : (batch, heads, seq_len_k, d_k)
    # output  : (batch, heads, seq_len_q, d_k)
    output = torch.matmul(weights, V)

    return output, weights


# ── CONCRETE EXAMPLE ──────────────────────────────────────────────────────────
torch.manual_seed(42)

batch, heads, seq_len, d_k = 1, 1, 3, 4

Q = torch.randn(batch, heads, seq_len, d_k)
K = torch.randn(batch, heads, seq_len, d_k)
V = torch.randn(batch, heads, seq_len, d_k)

# sequence: ["Rohit", "learns", <PAD>]
# mask 1 = real token, 0 = padding
raw_mask = torch.tensor([[1, 1, 0]])           # (1, 3)
mask_4d  = raw_mask.unsqueeze(1).unsqueeze(2)  # (1, 1, 1, 3) — broadcasts over heads and query positions

output, weights = scaled_dot_product_attention(Q, K, V, mask_4d)

print("shapes")
print("Q              :", Q.shape)
print("K              :", K.shape)
print("V              :", V.shape)
print("mask_4d        :", mask_4d.shape)
print("output         :", output.shape)
print("weights        :", weights.shape)
# shapes
# Q              : torch.Size([1, 1, 3, 4])
# K              : torch.Size([1, 1, 3, 4])
# V              : torch.Size([1, 1, 3, 4])
# mask_4d        : torch.Size([1, 1, 1, 3])
# output         : torch.Size([1, 1, 3, 4])
# weights        : torch.Size([1, 1, 3, 3])

print("\nattention weights (rows = queries, cols = keys)")
print(weights[0, 0])
# each row sums to 1.0 (softmax)
# column 2 (PAD position) ≈ 0.0 for all query rows

print("\ncolumn sums (should be ~1.0 per row, not per col)")
print("row sums :", weights[0, 0].sum(dim=-1))
# row sums : tensor([1.0000, 1.0000, 1.0000])

print("\npad column (col 2) weights ≈ 0")
print("col 2   :", weights[0, 0, :, 2])
# col 2   : tensor([~0.0, ~0.0, ~0.0])


# ── WHY √d_k SCALING MATTERS ─────────────────────────────────────────────────
# Without scaling, dot products grow with d_k → pushes softmax into flat/peaked zones
# → vanishing gradients in training
# Demonstrating the effect:

Q_demo = torch.randn(1, 1, 4, 64)   # d_k = 64
K_demo = torch.randn(1, 1, 4, 64)

scores_unscaled = torch.matmul(Q_demo, K_demo.transpose(-2, -1))
scores_scaled   = scores_unscaled / math.sqrt(64)

print("\nscaling effect (d_k=64)")
print("unscaled score std :", scores_unscaled.std().item())
print("scaled   score std :", scores_scaled.std().item())
# unscaled std ≈ 8.0  (≈ √64 times larger)
# scaled   std ≈ 1.0  (variance controlled)

w_unscaled = F.softmax(scores_unscaled, dim=-1)
w_scaled   = F.softmax(scores_scaled,   dim=-1)

print("unscaled softmax entropy (lower = more peaked) :", -(w_unscaled * (w_unscaled + 1e-9).log()).sum(-1).mean().item())
print("scaled   softmax entropy (higher = more spread):", -(w_scaled   * (w_scaled   + 1e-9).log()).sum(-1).mean().item())
# unscaled entropy lower  → attention collapses onto one token → gradient dies
# scaled   entropy higher → attention spread across positions → healthy gradients


# ── SMALL MANUAL WORKED EXAMPLE ──────────────────────────────────────────────
# Tiny values so you can verify by hand: seq_len=2, d_k=2, no mask

Q_m = torch.tensor([[[[ 1.0,  0.0],   # query for position 0
                       [ 0.0,  1.0]]]])  # query for position 1
K_m = torch.tensor([[[[ 1.0,  0.0],
                       [ 0.0,  1.0]]]])
V_m = torch.tensor([[[[ 10.0, 20.0],
                       [ 30.0, 40.0]]]])

out_m, w_m = scaled_dot_product_attention(Q_m, K_m, V_m)

print("\nmanual example")
print("raw scores before scale")
raw = torch.matmul(Q_m, K_m.transpose(-2, -1))
print(raw[0, 0])
# [[1/√2, 0/√2],   → Q[0]·K[0]=1, Q[0]·K[1]=0
#  [0/√2, 1/√2]]   → Q[1]·K[0]=0, Q[1]·K[1]=1

print("weights after softmax")
print(w_m[0, 0])
# position 0 attends mostly to key 0
# position 1 attends mostly to key 1

print("output")
print(out_m[0, 0])
# position 0 → blend of V rows weighted toward row 0 → close to [10, 20]
# position 1 → blend of V rows weighted toward row 1 → close to [30, 40]


# ── DRY RUN : scaled_dot_product_attention on ["Rohit", "learns", <PAD>] ─────
# State  : Q, K, V shape (1,1,3,4); mask_4d = [[[[ 1, 1, 0 ]]]]
#
# Step 1 — scores = Q @ Kᵀ / √4   shape (1,1,3,3)
#           entry [0,0,i,j] = dot(Q[i], K[j]) / 2.0
#
# Step 2 — masked_fill: scores[:,:,:,2] = -1e9
#           column 2 (PAD key) forced to -∞ before softmax
#
# Step 3 — softmax(scores, dim=-1)
#           each row: exp(-1e9) / (exp(s0) + exp(s1) + exp(-1e9))
#                   ≈ 0 for column 2 in every row
#           weights[:,:,:,2] ≈ 0.0
#
# Step 4 — output = weights @ V   shape (1,1,3,4)
#           PAD value vector V[:,2,:] gets weight ≈ 0 → contributes nothing
#
# New state : output[0,0] = contextual vector per real token
#             PAD token output exists but is discarded in downstream masking

shapes
Q              : torch.Size([1, 1, 3, 4])
K              : torch.Size([1, 1, 3, 4])
V              : torch.Size([1, 1, 3, 4])
mask_4d        : torch.Size([1, 1, 1, 3])
output         : torch.Size([1, 1, 3, 4])
weights        : torch.Size([1, 1, 3, 3])

attention weights (rows = queries, cols = keys)
tensor([[0.4934, 0.5066, 0.0000],
        [0.3920, 0.6080, 0.0000],
        [0.5616, 0.4384, 0.0000]])

column sums (should be ~1.0 per row, not per col)
row sums : tensor([1.0000, 1.0000, 1.0000])

pad column (col 2) weights ≈ 0
col 2   : tensor([0., 0., 0.])

scaling effect (d_k=64)
unscaled score std : 6.114415645599365
scaled   score std : 0.7643019556999207
unscaled softmax entropy (lower = more peaked) : 0.23715852200984955
scaled   softmax entropy (higher = more spread): 1.1559627056121826

manual example
raw scores before scale
tensor([[1., 0.],
        [0., 1.]])
weights after softmax
tensor([[0.6698, 0.3302],
        [0.3302, 0.6698]])
output
tensor([[16.6048, 26.6048],
   

In [3]:
import torch
import torch.nn as nn

# ── Step 1: Define a simple char-level tokenizer ─────────────────────────────
class SimpleTokenizer:
    def __init__(self, corpus, pad_token="<PAD>", unk_token="<UNK>"):
        chars = sorted(set(corpus))
        self.vocab = {ch: i+2 for i, ch in enumerate(chars)}  # start at 2
        self.vocab[pad_token] = 0
        self.vocab[unk_token] = 1
        self.inv_vocab = {v: k for k, v in self.vocab.items()}
        self.pad_id = 0
        self.unk_id = 1

    def encode(self, text):
        return [self.vocab.get(ch, self.unk_id) for ch in text]

    def decode(self, ids, skip_specials=True):
        chars = []
        for id_ in ids:
            token = self.inv_vocab.get(id_, "<UNK>")
            if skip_specials and id_ in (self.pad_id,):
                continue
            chars.append(token)
        return "".join(chars)


# ── Step 2: Batch encoding with padding + attention mask ─────────────────────
def batch_encode(tokenizer, texts, max_length=None):
    encoded = [tokenizer.encode(t) for t in texts]
    if max_length is None:
        max_length = max(len(e) for e in encoded)
    input_ids       = []
    attention_masks = []
    for seq in encoded:
        seq     = seq[:max_length]
        pad_len = max_length - len(seq)
        mask    = [1] * len(seq) + [0] * pad_len
        seq     = seq + [tokenizer.pad_id] * pad_len
        input_ids.append(seq)
        attention_masks.append(mask)
    return (
        torch.tensor(input_ids,       dtype=torch.long),
        torch.tensor(attention_masks, dtype=torch.long)
    )


# ── Step 3: Embedding lookup + masking ───────────────────────────────────────
class TransformerInputLayer(nn.Module):
    def __init__(self, vocab_size, d_model, max_seq_len):
        super().__init__()
        self.token_emb    = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.position_emb = nn.Embedding(max_seq_len, d_model)
        self.norm         = nn.LayerNorm(d_model)

    def forward(self, input_ids, attention_mask):
        B, T      = input_ids.shape
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0)
        tok_emb   = self.token_emb(input_ids)
        pos_emb   = self.position_emb(positions)
        x         = self.norm(tok_emb + pos_emb)
        mask_expanded = attention_mask.unsqueeze(-1).float()
        x = x * mask_expanded
        return x


In [4]:
# ═════════════════════════════════════════════════════════════════════════════
# ── DRY RUN — corpus = "Hello, I am learning Transformers! Great."
# ═════════════════════════════════════════════════════════════════════════════


# ── STAGE 1 : SimpleTokenizer.__init__(corpus) ────────────────────────────────
#
# corpus = "Hello, I am learning Transformers! Great."
#
# Step A — set(corpus) : unique characters (order arbitrary)
#   { 'H','e','l','o',',',' ','I','a','m','r','n','g','T','s','f','!','G','t','.' }
#
# Step B — sorted(set(corpus)) : alphabetical (ASCII order)
#   [' ', '!', ',', '.', 'G', 'H', 'I', 'T', 'a', 'e', 'f', 'g',
#    'i', 'l', 'm', 'n', 'o', 'r', 's', 't']
#   → 20 unique chars
#
# Step C — build vocab dict  (IDs start at 2 ; 0 and 1 reserved for specials)
#   char   id
#   ' '  →  2
#   '!'  →  3
#   ','  →  4
#   '.'  →  5
#   'G'  →  6
#   'H'  →  7
#   'I'  →  8
#   'T'  →  9
#   'a'  → 10
#   'e'  → 11
#   'f'  → 12
#   'g'  → 13
#   'i'  → 14
#   'l'  → 15
#   'm'  → 16
#   'n'  → 17
#   'o'  → 18
#   'r'  → 19
#   's'  → 20
#   't'  → 21
#
# Step D — add special tokens
#   '<PAD>' → 0    (self.pad_id = 0)
#   '<UNK>' → 1    (self.unk_id = 1)
#
# Final vocab size = 20 chars + 2 specials = 22 entries
#
# Step E — inv_vocab (reverse lookup, used by decode())
#   0→'<PAD>', 1→'<UNK>', 2→' ', 3→'!', 4→',', 5→'.', 6→'G',
#   7→'H', 8→'I', 9→'T', 10→'a', 11→'e', 12→'f', 13→'g',
#   14→'i', 15→'l', 16→'m', 17→'n', 18→'o', 19→'r', 20→'s', 21→'t'


# ── STAGE 2 : tokenizer.encode(text) ─────────────────────────────────────────
#
# encode() walks each character and does vocab.get(ch, unk_id)
# Unknown char → returns unk_id = 1
#
# ── encode("Hello,") ─────────────────────────────────────────────────────────
#
#   ch   vocab.get(ch, 1)
#   'H'  →  7
#   'e'  → 11
#   'l'  → 15
#   'l'  → 15
#   'o'  → 18
#   ','  →  4
#
#   result : [7, 11, 15, 15, 18, 4]   length = 6
#
# ── encode("I am learning!") ─────────────────────────────────────────────────
#
#   ch   vocab.get(ch, 1)
#   'I'  →  8
#   ' '  →  2
#   'a'  → 10
#   'm'  → 16
#   ' '  →  2
#   'l'  → 15
#   'e'  → 11
#   'a'  → 10
#   'r'  → 19
#   'n'  → 17
#   'i'  → 14
#   'n'  → 17
#   'g'  → 13
#   '!'  →  3
#
#   result : [8, 2, 10, 16, 2, 15, 11, 10, 19, 17, 14, 17, 13, 3]   length = 14


# ── STAGE 3 : batch_encode(tokenizer, texts) ─────────────────────────────────
#
# texts = ["Hello,", "I am learning!"]
#
# After encode() on both:
#   encoded[0] = [7, 11, 15, 15, 18, 4]               len = 6
#   encoded[1] = [8, 2, 10, 16, 2, 15, 11, 10, 19, 17, 14, 17, 13, 3]  len = 14
#
# max_length = max(6, 14) = 14   (no truncation needed for either sequence)
#
# ── Iteration 1 : seq = encoded[0]  (len=6) ──────────────────────────────────
#
#   seq     = [7, 11, 15, 15, 18, 4]   (no truncation, 6 ≤ 14)
#   pad_len = 14 − 6 = 8
#   mask    = [1, 1, 1, 1, 1, 1,  0, 0, 0, 0, 0, 0, 0, 0]
#              ←— real tokens —→  ←———— padding zeros ————→
#   seq     = [7, 11, 15, 15, 18, 4,  0, 0, 0, 0, 0, 0, 0, 0]
#              ←— real tokens —→  ←— pad_id=0 appended ×8 →
#
# ── Iteration 2 : seq = encoded[1]  (len=14) ─────────────────────────────────
#
#   seq     = [8, 2, 10, 16, 2, 15, 11, 10, 19, 17, 14, 17, 13, 3]
#   pad_len = 14 − 14 = 0
#   mask    = [1, 1,  1,  1, 1,  1,  1,  1,  1,  1,  1,  1,  1, 1]
#   seq     = [8, 2, 10, 16, 2, 15, 11, 10, 19, 17, 14, 17, 13, 3]
#              (unchanged — no padding required)
#
# ── torch.tensor() conversion ────────────────────────────────────────────────
#
#   input_ids shape      : [2, 14]   (batch=2, seq_len=14)
#   attention_mask shape : [2, 14]
#
#   input_ids:
#   tensor([[ 7, 11, 15, 15, 18,  4,  0,  0,  0,  0,  0,  0,  0,  0],
#            [ 8,  2, 10, 16,  2, 15, 11, 10, 19, 17, 14, 17, 13,  3]])
#
#   attention_mask:
#   tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
#            [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


# ── STAGE 4 : TransformerInputLayer.__init__() ────────────────────────────────
#
#   vocab_size  = 22   (20 chars + 2 specials)
#   d_model     = 16
#   max_seq_len = 32
#
#   token_emb    : nn.Embedding(22, 16, padding_idx=0)
#     → weight matrix shape [22, 16]
#     → row 0 (pad_id) is fixed at all-zeros, never updated by gradients
#
#   position_emb : nn.Embedding(32, 16)
#     → weight matrix shape [32, 16]
#     → row i gives the learned positional vector for position i
#
#   norm         : nn.LayerNorm(16)
#     → learnable γ (scale) and β (shift), both shape [16]
#     → normalises last dimension (d_model) to mean=0, std=1, then rescales


# ── STAGE 5 : TransformerInputLayer.forward() ────────────────────────────────
#
#   input_ids      shape : [2, 14]   B=2, T=14
#   attention_mask shape : [2, 14]
#
# ── Step A : positions = torch.arange(14).unsqueeze(0) ───────────────────────
#
#   torch.arange(14) = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
#   .unsqueeze(0)    → shape [1, 14]   (broadcast over batch dimension)
#
# ── Step B : tok_emb = token_emb(input_ids) ──────────────────────────────────
#
#   Each integer in input_ids is used as a row index into the [22, 16] embedding
#   matrix. The lookup is a simple table read — no arithmetic.
#
#   Batch 0 (row-by-row):
#     pos 0 : id= 7 → embedding_matrix[7]   → vector of shape [16]   ('H')
#     pos 1 : id=11 → embedding_matrix[11]  → vector of shape [16]   ('e')
#     pos 2 : id=15 → embedding_matrix[15]  → vector of shape [16]   ('l')
#     pos 3 : id=15 → embedding_matrix[15]  → SAME vector as pos 2   ('l')
#     pos 4 : id=18 → embedding_matrix[18]  → vector of shape [16]   ('o')
#     pos 5 : id= 4 → embedding_matrix[4]   → vector of shape [16]   (',')
#     pos 6–13: id= 0 → embedding_matrix[0] → all-zeros (padding_idx) ('PAD')
#
#   Batch 1 (no padding):
#     pos 0 : id= 8 → embedding_matrix[8]   → vector of shape [16]   ('I')
#     pos 1 : id= 2 → embedding_matrix[2]   → vector of shape [16]   (' ')
#     ... (every id maps to its own 16-dim row)
#     pos 13: id= 3 → embedding_matrix[3]   → vector of shape [16]   ('!')
#
#   tok_emb shape : [2, 14, 16]
#
# ── Step C : pos_emb = position_emb(positions) ───────────────────────────────
#
#   positions shape : [1, 14]
#   Lookup: row i of position embedding matrix for each position 0..13
#
#   pos  0 → position_matrix[0]  → [16]
#   pos  1 → position_matrix[1]  → [16]
#   ...
#   pos 13 → position_matrix[13] → [16]
#
#   pos_emb shape : [1, 14, 16]
#   → broadcasts to [2, 14, 16] when added to tok_emb
#     (same positional vectors added to every sample in the batch)
#
# ── Step D : x = LayerNorm(tok_emb + pos_emb) ────────────────────────────────
#
#   tok_emb + pos_emb → element-wise add → shape [2, 14, 16]
#
#   LayerNorm normalises over the last dimension (d_model=16) independently
#   for each (batch, position) pair:
#
#     For each token vector v of shape [16]:
#       μ    = mean(v)                         scalar
#       σ²   = var(v)                          scalar
#       v̂   = (v − μ) / √(σ² + ε)            shape [16], mean≈0, std≈1
#       out  = γ ⊙ v̂ + β                      shape [16], γ,β learned
#
#   x shape after LayerNorm : [2, 14, 16]
#
#   NOTE: Padding positions (id=0) have tok_emb=zeros but pos_emb≠zeros,
#         so they are NOT zero here yet — that is handled in Step E.
#
# ── Step E : mask_expanded and zeroing pad positions ─────────────────────────
#
#   attention_mask shape           : [2, 14]
#   .unsqueeze(-1)                 : [2, 14, 1]
#   .float()                       : dtype float32, values still 0.0 or 1.0
#
#   x = x * mask_expanded          → broadcast [2,14,16] * [2,14,1]
#     → each of the 16 features at a padding position is multiplied by 0.0
#     → each of the 16 features at a real-token position is multiplied by 1.0
#
#   Effect on Batch 0 (pad positions 6–13):
#     pos 0–5 : kept as-is  (mask = 1)
#     pos 6–13: all 16 dims → 0.0  (mask = 0)
#
#   Effect on Batch 1 (no padding):
#     pos 0–13: kept as-is  (mask = 1 everywhere)
#
#   x shape : [2, 14, 16]   ← final output


# ── STAGE 6 : Demo — final tensor shapes & values ────────────────────────────
#
#   output = layer(input_ids, attention_mask)
#
#   output.shape = [2, 14, 16]
#                   │   │   └── d_model : 16 features per token
#                   │   └────── seq_len : 14 (padded to longest sequence)
#                   └────────── batch   :  2 sentences
#
#   output[0, 0:6, :]  → non-zero vectors  (real tokens of "Hello,")
#   output[0, 6:14, :] → all-zero vectors  (padding, zeroed by mask)
#   output[1, 0:14, :] → all non-zero      (no padding in "I am learning!")
#
# ── WHY ZERO OUT PADDING? ─────────────────────────────────────────────────────
#
#   padding_idx=0 in nn.Embedding zeroes only the embedding row for id=0.
#   But after adding positional embeddings, those positions are no longer zero.
#   The mask multiply restores them to zero so downstream layers (attention,
#   pooling, etc.) cannot "see" padding as real signal.
#   The canonical way to do this in attention layers is via an additive mask
#   (−∞ before softmax), but zeroing the input embeddings is good practice
#   for the input layer specifically.
#
# ── COMPLEXITY SUMMARY ────────────────────────────────────────────────────────
#
#   Tokenizer build   : O(|corpus|) time, O(V) space where V = unique chars
#   encode(text)      : O(|text|)   — one dict lookup per character
#   batch_encode      : O(B · T)    — B sequences, T max length
#   forward() pass    :
#     token_emb lookup  O(B · T)    — index into embedding table
#     pos_emb  lookup   O(T)        — same positions for all batch items
#     LayerNorm         O(B · T · d_model)
#     mask multiply     O(B · T · d_model)
#   Total forward     : O(B · T · d_model)



In [5]:
corpus = "Hello, I am learning Transformers! Great."
tokenizer = SimpleTokenizer(corpus)

texts = ["Hello,", "I am learning!"]
input_ids, attention_mask = batch_encode(tokenizer, texts)

print("input_ids:\n", input_ids)
print("\nattention_mask:\n", attention_mask)

layer = TransformerInputLayer(
    vocab_size=len(tokenizer.vocab),
    d_model=16,
    max_seq_len=32
)

output = layer(input_ids, attention_mask)
print("\nOutput shape:", output.shape)   # torch.Size([2, 14, 16])

input_ids:
 tensor([[ 7, 11, 15, 15, 18,  4,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 8,  2, 10, 16,  2, 15, 11, 10, 19, 17, 14, 17, 13,  3]])

attention_mask:
 tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

Output shape: torch.Size([2, 14, 16])


In [6]:
import torch
import torch.nn.functional as F
import math

def attention_with_mask_demo():
    seq_len = 4
    d_k     = 2

    torch.manual_seed(42)
    Q = torch.randn(1, seq_len, d_k)
    K = torch.randn(1, seq_len, d_k)
    V = torch.randn(1, seq_len, d_k)

    attention_mask = torch.tensor([[1, 1, 0, 0]], dtype=torch.float)
    # shape: [1, 4]

    scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(d_k)
    mask   = attention_mask.unsqueeze(1)
    scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    output  = torch.bmm(weights, V)
    print("Output shape:", output.shape)


# ═════════════════════════════════════════════════════════════════════════════
# ── DRY RUN  (seed=42, seq_len=4, d_k=2, batch=1)
# ═════════════════════════════════════════════════════════════════════════════
#
# Tokens (logical): [Hello, world, <PAD>, <PAD>]
# attention_mask  : [  1,     1,     0,     0  ]   shape [1, 4]
#   1 = real token → attend normally
#   0 = padding    → must be blocked from receiving or contributing attention


# ── STAGE 1 : Q, K, V after torch.randn (seed=42) ────────────────────────────
#
# Each is shape [1, 4, 2]  — batch=1, 4 tokens, d_k=2 features
#
#          feat_0   feat_1
# Q[0] = [[ 0.335,  1.525],    ← Hello
#          [ 0.108,  0.674],    ← world
#          [-1.116, -0.530],    ← <PAD>
#          [ 0.612,  0.222]]    ← <PAD>
#
#          feat_0   feat_1
# K[0] = [[-0.603, -0.559],
#          [ 0.399,  0.258],
#          [-0.074,  0.929],
#          [ 1.101, -0.182]]
#
#          feat_0   feat_1
# V[0] = [[ 0.473, -0.384],
#          [-0.574,  0.685],
#          [-0.118,  1.237],
#          [ 0.213, -0.472]]


# ── STAGE 2 : Raw scores  =  Q @ Kᵀ  /  √d_k ────────────────────────────────
#
# K.transpose(1,2) : shape [1, 2, 4]  — columns become rows
# bmm(Q, Kᵀ)       : shape [1, 4, 4]  — each [i,j] = dot(Q[i], K[j])
#
# scores[i][j] = (Q[i] · K[j]) / √2     where √2 ≈ 1.414
#
# Why divide by √d_k?
#   Dot products grow with d_k; dividing keeps pre-softmax values in a range
#   where softmax gradients don't vanish (prevents extreme peakiness).
#
# Approximate raw score matrix (before masking):
#
#          K_Hello  K_world  K_PAD0  K_PAD1
# Q_Hello  [ -0.64,   0.53,   1.05,   0.34 ]
# Q_world  [ -0.30,   0.16,   0.44,   0.06 ]
# Q_PAD0   [  0.44,  -0.14,  -0.27,  -1.15 ]  ← PAD row — will be zeroed later
# Q_PAD1   [ -0.45,   0.31,   0.53,   0.39 ]  ← PAD row — will be zeroed later
#
# shape: [1, 4, 4]


# ── STAGE 3 : Expand mask and apply masked_fill ───────────────────────────────
#
# attention_mask          : [1, 4]       values 1,1,0,0
# .unsqueeze(1)           : [1, 1, 4]    insert query dimension
#   → broadcasts to       : [1, 4, 4]    same mask column applied to every query row
#
# masked_fill(mask == 0, -inf)
#   Wherever mask == 0  (columns 2 and 3 = PAD positions):
#     score → -inf
#   Wherever mask == 1  (columns 0 and 1 = real tokens):
#     score unchanged
#
# After masking:
#          K_Hello  K_world  K_PAD0    K_PAD1
# Q_Hello  [ -0.64,   0.53,   -inf,    -inf ]
# Q_world  [ -0.30,   0.16,   -inf,    -inf ]
# Q_PAD0   [  0.44,  -0.14,   -inf,    -inf ]
# Q_PAD1   [ -0.45,   0.31,   -inf,    -inf ]
#
# Key insight:
#   We mask COLUMNS (key positions), not rows.
#   This means: every query — including PAD queries — is blocked
#   from attending to PAD keys. PAD tokens produce no signal.


# ── STAGE 4 : Softmax over last dim (dim=-1) ─────────────────────────────────
#
# softmax([a, b, -inf, -inf]) over 4 elements:
#   e^(-inf) = 0  →  those columns contribute 0 to the sum
#   remaining mass shared only among real-token columns (0 and 1)
#
# Per-row calculation (approximate):
#
#   Q_Hello : softmax([-0.64, 0.53, -inf, -inf])
#     e^-0.64 ≈ 0.527,  e^0.53 ≈ 1.699,  sum ≈ 2.226
#     → [0.527/2.226,  1.699/2.226,  0,  0]  ≈ [0.24, 0.76, 0.00, 0.00]
#
#   Q_world : softmax([-0.30, 0.16, -inf, -inf])
#     e^-0.30 ≈ 0.741,  e^0.16 ≈ 1.173,  sum ≈ 1.914
#     → [0.741/1.914,  1.173/1.914,  0,  0]  ≈ [0.39, 0.61, 0.00, 0.00]
#
#   Q_PAD0  : softmax([ 0.44,-0.14, -inf, -inf])
#     → still sums to 1 over cols 0,1 (PAD queries don't attend to PAD keys)
#     ≈ [0.65, 0.35, 0.00, 0.00]
#
#   Q_PAD1  : softmax([-0.45, 0.31, -inf, -inf])
#     ≈ [0.32, 0.68, 0.00, 0.00]
#
# weights shape: [1, 4, 4]   all rows sum to 1.0, PAD columns are exactly 0.0


# ── STAGE 5 : Weighted sum of V  =  weights @ V ───────────────────────────────
#
# bmm(weights, V) : [1, 4, 4] @ [1, 4, 2] → [1, 4, 2]
#
# output[i] = Σ_j  weights[i][j] * V[j]
#
# For Q_Hello row (weights ≈ [0.24, 0.76, 0, 0]):
#   output = 0.24 * V[Hello] + 0.76 * V[world] + 0*V[PAD] + 0*V[PAD]
#          = 0.24*[ 0.473,-0.384] + 0.76*[-0.574, 0.685]
#          ≈ [ 0.114,-0.092] + [-0.436, 0.521]
#          ≈ [-0.322,  0.429]
#
# PAD output rows are computed but carry no meaning downstream —
# the upstream embedding mask (Stage 5 of TransformerInputLayer) already
# zeroed PAD embeddings; here PAD queries produce outputs that are simply
# ignored by any subsequent pooling or classification head.
#
# output shape: [1, 4, 2]   ── [batch, seq_len, d_k]


# ── SHAPE FLOW SUMMARY ────────────────────────────────────────────────────────
#
#   Q, K, V              [1, 4, 2]
#   K.transpose(1,2)     [1, 2, 4]
#   bmm(Q, Kᵀ)           [1, 4, 4]   raw scores
#   / √d_k               [1, 4, 4]   scaled scores
#   mask unsqueeze(1)    [1, 1, 4]  → broadcast [1, 4, 4]
#   masked_fill          [1, 4, 4]   PAD columns = -inf
#   softmax(dim=-1)      [1, 4, 4]   PAD columns = 0.0, rows sum to 1
#   bmm(weights, V)      [1, 4, 2]   final attended output


attention_with_mask_demo()

Output shape: torch.Size([1, 4, 2])
